# 03 — Preprocessing

Goal: produce two parallel, directly comparable datasets — one with all 30 features and one with the feature subset selected in `02_feature_engineering.ipynb` — scaled and split, ready for `04_modeling.ipynb`.

> Class-imbalance handling (SMOTE, oversampling, undersampling, `class_weight`) is **not** applied here. It's compared as a variable in `04_modeling.ipynb` instead, so resampling only ever happens inside each cross-validation training fold (avoiding leakage) rather than being baked into a single choice.

In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

os.makedirs('../data/processed', exist_ok=True)

## 1. Load Data and Feature Sets

In [2]:
raw = pd.read_csv('../data/raw/creditcard.csv')

with open('../data/processed/selected_features.json') as f:
    selected_features = json.load(f)

all_features = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Time']

feature_sets = {
    'full': all_features,
    'selected': selected_features,
}

for name, features in feature_sets.items():
    print(f'{name}: {len(features)} features')

full: 30 features
selected: 28 features


## 2. Scale, Split, and Export Each Variant

`V1`–`V28` are already PCA-standardized; only `Amount`/`Time` need scaling, and only if they're present in a given feature set. Both variants use the same `random_state` and row count, so `train_test_split` assigns identical rows to train/test in each — the two variants stay directly comparable.

In [3]:
for name, features in feature_sets.items():
    df = raw[features + ['Class']].copy()

    scaler = StandardScaler()
    for col in ['Amount', 'Time']:
        if col in df.columns:
            df[f'{col}_scaled'] = scaler.fit_transform(df[[col]])
            df = df.drop(columns=[col])

    X = df.drop(columns=['Class'])
    y = df['Class']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    X_train.assign(Class=y_train.values).to_csv(f'../data/processed/train_{name}.csv', index=False)
    X_test.assign(Class=y_test.values).to_csv(f'../data/processed/test_{name}.csv', index=False)

    print(f'[{name}] train: {X_train.shape}, test: {X_test.shape}, '
          f'train fraud rate: {y_train.mean():.4f}, test fraud rate: {y_test.mean():.4f}')
    print(f'  Saved: data/processed/train_{name}.csv, data/processed/test_{name}.csv')

[full] train: (227845, 30), test: (56962, 30), train fraud rate: 0.0017, test fraud rate: 0.0017
  Saved: data/processed/train_full.csv, data/processed/test_full.csv
[selected] train: (227845, 28), test: (56962, 28), train fraud rate: 0.0017, test fraud rate: 0.0017
  Saved: data/processed/train_selected.csv, data/processed/test_selected.csv
